# 06 — Bộ chấm v1.3 và thí nghiệm cơ chế của nhãn phiên bản

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

| Notebook | Việc | Đầu ra |
|---|---|---|
| `04_pilot_llm_rag` | Pilot: kho trộn phiên bản làm tỉ lệ trả lời lỗi thời tăng mạnh | `results/pilot-18.1-19.2/` |
| `05_ngu_canh_kiem_soat` | Nguyên nhân chính là **sự có mặt** của đoạn cũ; **nhãn phiên bản** giảm lỗi thời rất mạnh | `results/ctx-18.1-19.2/` |
| **`06_co_che_nhan_phien_ban` (notebook này)** | (A) bộ chấm v1.3; (B) chấm lại 04, 05; (C) **vì sao** nhãn phiên bản có tác dụng | `results/mech-18.1-19.2/` |

## Phần A–B: bộ chấm v1.3

So sánh với nhãn chấm tay cho thấy hai điểm bộ chấm v1.2 làm khác người chấm:

1. **ID đúng nhưng tên lỗi thời**, ví dụ `"TA0005 Defense Evasion"` khi v19.2 đã gọi là Stealth. Bản v1.2 chỉ bật cờ `label_drift`; bản v1.3 tính là **lỗi thời** (`STALE`).
2. **Mã và tên không khớp nhau**, ví dụ `"TA0005 Impact"`, `"TA0112 Defense Evasion"`, hoặc mã không tồn tại như `TA0118`. Bản v1.2 chỉ xét mã. Bản v1.3 xếp các câu này vào nhãn riêng **`INCONSISTENT`** (không nhất quán), không ép vào "đúng" hay "lỗi thời".

Bản v1.3 **không thay** bản v1.2: chỉ số chính của notebook 05 (`drift_ext`, chỉ xét mã) giữ nguyên. Bản v1.3 cho thêm một **phép chấm chặt** để làm phân tích độ nhạy: `stale13` = lỗi thời theo mã **hoặc** theo tên, và câu trả lời phải nhất quán mã–tên.

Phần B chấm lại toàn bộ câu trả lời đã có của notebook 04 và 05 bằng v1.3, **không cần GPU**, để xem kết luận có đổi không.

## Phần C: nhãn phiên bản có tác dụng nhờ cơ chế nào?

Notebook 05 cho thấy gắn nhãn `[Source: MITRE ATT&CK Enterprise v19.2 (current release)]` làm lỗi thời giảm từ ~0,9 xuống ~0,05–0,27. Nhưng câu hỏi cũng chứa chuỗi `(v19.2)`, còn nhãn chứa chữ *current* và *superseded*. Có thể mô hình chỉ **khớp chữ** chứ không hiểu "bản nào mới hơn". Phần C tách các khả năng đó.

### Mười điều kiện ngữ cảnh

Mọi điều kiện đều chứa **cả** đoạn cũ (O) và đoạn mới (N), cùng các đoạn nhiễu như notebook 05 (tổng 4 đoạn).

| Điều kiện | Thứ tự | Nhãn trên đoạn | Mục đích |
|---|---|---|---|
| `old_new`, `new_old` | O,N / N,O | không | mốc so sánh (lặp lại notebook 05) |
| `tag_old_new`, `tag_new_old` | O,N / N,O | **đầy đủ**: `v18.1 (superseded)` / `v19.2 (current release)` | lặp lại notebook 05 |
| `vtag_old_new`, `vtag_new_old` | O,N / N,O | **chỉ số phiên bản**: `v18.1` / `v19.2` | bỏ chữ *current/superseded* |
| `swap_old_new`, `swap_new_old` | O,N / N,O | **tráo nhãn**: đoạn cũ ghi `v19.2 (current release)`, đoạn mới ghi `v18.1 (superseded)` | mô hình theo **nhãn** hay theo **nội dung**? |
| `bridge_old_new`, `bridge_new_old` | O,N,B / N,O,B | đầy đủ | thêm **ghi chú cầu nối** B cho mã bị thu hồi |

- **Ghi chú cầu nối (B)** chỉ có ở câu có mã bị thu hồi: *"In MITRE ATT&CK v19, technique T1562.004 (…) was revoked and replaced by T1686 (…)."* Câu không có mã thu hồi thì chỗ đó là đoạn nhiễu, nên ngữ cảnh giống hệt `tag_*`.
- **Tráo nhãn** chỉ áp cho hai đoạn bằng chứng; đoạn nhiễu (nội dung v19.2) giữ nhãn `v19.2 (current release)`.

### Hai cách hỏi

Mỗi câu hỏi được hỏi theo hai cách:

- `ver`: câu hỏi `current` của notebook 05, **có** `v19.2`. Ví dụ: *"According to the current MITRE ATT&CK Enterprise release (v19.2), which tactic(s)…"*
- `nover`: cùng câu hỏi nhưng **bỏ số phiên bản**. Ví dụ: *"According to the current MITRE ATT&CK Enterprise release, which tactic(s)…"*

Hai cách hỏi dùng **cùng** ngữ cảnh: cùng đoạn bằng chứng, cùng đoạn nhiễu.

**Lưu ý về D4.** Câu D1 và D2 vốn đã có chữ *current*, nên bản `nover` **chỉ** bỏ `(v19.2)`. Câu D4 lại viết *"As of MITRE ATT&CK Enterprise v19.2, …"*, không có chữ *current*. Bỏ số phiên bản thì phải thay bằng *"In the current MITRE ATT&CK Enterprise release, …"*, tức là thay đổi **hai** thứ cùng lúc. Vì vậy các so sánh giữa hai cách hỏi (M2, M2v, M3, M4n) lấy **D1 + D2 làm kết quả chính**; tính cả D4 chỉ để tham khảo.

### Giả thuyết (định trước, trước khi xem kết quả)

`m` = `drift_ext` với nhóm `changed` (tỉ lệ trả lời theo bản cũ, chỉ xét mã, như notebook 05) và `label_drift` với nhóm `label` (dùng tên tactic cũ). Mỗi so sánh làm riêng cho hai thứ tự X ∈ {`old_new`, `new_old`}, ghép cặp trên cùng câu hỏi.

Ký hiệu `rev(X)` là thứ tự ngược của X: `rev(old_new) = new_old`.

| | Câu hỏi | So sánh | Dự đoán | Dữ liệu |
|---|---|---|---|---|
| **M1** | Chữ *current/superseded* trong nhãn có cần không? | m(`vtag_X`, ver) − m(`tag_X`, ver) | > 0 nếu cần | mọi mẫu |
| **M2** | Với nhãn đầy đủ, số phiên bản trong câu hỏi có cần không? | m(`tag_X`, nover) − m(`tag_X`, ver) | > 0 nếu cần | D1 + D2 |
| **M2v** | Với nhãn chỉ có số, mô hình có dựa vào việc **khớp chữ** "v19.2" giữa câu hỏi và nhãn không? | m(`vtag_X`, nover) − m(`vtag_X`, ver) | > 0 nếu có | D1 + D2 |
| **M3** | Chỉ có số phiên bản trần, không gợi ý chữ nào: mô hình có tự suy ra v19.2 mới hơn v18.1? | m(`X`, nover) − m(`vtag_X`, nover) | > 0 nếu có | D1 + D2 |
| **M4** | Mô hình theo **nhãn** hay theo **nội dung**? Giữ nguyên thứ tự nhãn, chỉ tráo nội dung. | m(`swap_X`, ver) − m(`tag_rev(X)`, ver) | > 0 nếu theo nhãn | mọi mẫu |
| **M4n** | Như M4, với câu hỏi không có số phiên bản | m(`swap_X`, nover) − m(`tag_rev(X)`, nover) | > 0 nếu theo nhãn | D1 + D2 |
| **M4b** | Nhãn sai có tệ hơn không nhãn? | m(`swap_X`, ver) − m(`X`, ver) | > 0 nếu tệ hơn | mọi mẫu |
| **M5** | Ghi chú cầu nối có sửa được mã bị thu hồi không? | m(`tag_X`, ver) − m(`bridge_X`, ver), và acc(`bridge_X`) − acc(`tag_X`) | cả hai > 0 | `changed` có mã thu hồi |

**Vì sao M4 so với `tag_rev(X)`.** `swap_old_new` và `tag_new_old` có **cùng** dãy nhãn: đoạn 1 ghi *current*, đoạn 2 ghi *superseded*. Chúng chỉ khác ở chỗ nội dung nào mang nhãn nào. Notebook 05 cho thấy vị trí của nhãn có ảnh hưởng, nên so sánh này loại được yếu tố **vị trí nhãn**. Đổi lại, nó đảo vị trí **nội dung**, mà vị trí nội dung cũng có ảnh hưởng nhỏ (H1). Vì vậy bảng có thêm hai dòng:

- `swap_X − tag_X`: giữ nguyên thứ tự nội dung, để tham khảo.
- **M4 gộp hai thứ tự**: lấy trung bình trên cả hai thứ tự, nên cân bằng cả vị trí nhãn lẫn vị trí nội dung. Đây là dòng nên dùng làm kết luận chính cho M4.

**Giới hạn của M4b.** Không có nhãn thì lỗi thời đã ~0,8–0,9 (notebook 05), nên M4b chỉ còn ít "chỗ" để tăng. Kết quả M4b gần 0 **không** có nghĩa là nhãn sai vô hại.

**Vì sao M4 quan trọng cho an toàn.** Nếu mô hình theo nhãn hơn theo nội dung, kẻ tấn công, hoặc chỉ một lỗi gắn nhãn, có thể làm trợ lý trả lời sai chỉ bằng cách sửa siêu dữ liệu.

**Giới hạn của M5.** Chỉ có 40 câu gốc thuộc 15 cụm dòng dõi, nên khoảng tin cậy sẽ rộng. Với D4, ghi chú cầu nối gần như nói thẳng đáp án. Vì vậy M5 được báo cáo thêm theo từng mẫu ở mục 9.5.

**Lưu ý phương pháp.** Đây vẫn là thí nghiệm **khám phá cơ chế** trên lần chuyển phiên bản dùng để phát triển (v18.1 → v19.2). Số so sánh khá nhiều (khoảng 50 dòng mỗi mô hình), nên khi viết bài chỉ nhấn mạnh kết quả **nhất quán ở cả hai mô hình và hai thứ tự**. Kết luận chính thức sẽ được kiểm định lại trên lần chuyển phiên bản khác.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 12/10/2026 | Bản đầu tiên |

## 1. Cấu hình và kiểm tra đầu vào

Notebook này cần các đầu ra của notebook 02–05. Nếu có thư mục prompt của notebook 05, notebook này sẽ kiểm tra rằng các điều kiện lặp lại (`old_new`, `new_old`, `tag_*` với cách hỏi `ver`) có prompt **giống hệt** notebook 05.

In [1]:
from pathlib import Path
import os, sys, json, re, math, time, hashlib, subprocess, collections, datetime, zipfile, signal
import numpy as np
import pandas as pd

V_OLD, V_NEW = "18.1", "19.2"
ROOT    = Path.cwd()
DRIFT   = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
PILOT   = ROOT / "results" / f"pilot-{V_OLD}-{V_NEW}"     # notebook 04
CTX     = ROOT / "results" / f"ctx-{V_OLD}-{V_NEW}"       # notebook 05
OUT     = ROOT / "results" / f"mech-{V_OLD}-{V_NEW}"      # notebook này
PROMPTS = OUT / "prompts"
SMOKE_PROMPTS = OUT / "prompts_smoke"
RAW     = OUT / "raw"
LOGS    = OUT / "logs"
for p in (OUT, PROMPTS, SMOKE_PROMPTS, RAW, LOGS):
    p.mkdir(parents=True, exist_ok=True)

HF_HOME         = "/data/hf"
VLLM_CACHE_ROOT = "/data/vllm-cache"
MODELS = [
    dict(tag="qwen3-8b", repo="Qwen/Qwen3-8B-AWQ",
         revision="4da05a8edb55c6046cce958586c33b61da07bb79", released="2025-05",
         chat_template_kwargs={"enable_thinking": False}),
    dict(tag="llama3.1-8b", repo="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
         revision="db1f81ad4b8c7e39777509fac66c652eb0a52f91", released="2024-07",
         chat_template_kwargs={}),
]
# order: thứ tự bằng chứng; tag: kiểu nhãn (None / "full" / "ver"); swap: tráo nhãn hai đoạn bằng chứng
CONDITIONS = {
    "old_new":        dict(order=["O", "N"],      tag=None,   swap=False),
    "new_old":        dict(order=["N", "O"],      tag=None,   swap=False),
    "tag_old_new":    dict(order=["O", "N"],      tag="full", swap=False),
    "tag_new_old":    dict(order=["N", "O"],      tag="full", swap=False),
    "vtag_old_new":   dict(order=["O", "N"],      tag="ver",  swap=False),
    "vtag_new_old":   dict(order=["N", "O"],      tag="ver",  swap=False),
    "swap_old_new":   dict(order=["O", "N"],      tag="full", swap=True),
    "swap_new_old":   dict(order=["N", "O"],      tag="full", swap=True),
    "bridge_old_new": dict(order=["O", "N", "B"], tag="full", swap=False),
    "bridge_new_old": dict(order=["N", "O", "B"], tag="full", swap=False),
}
REPLICATED = ["old_new", "new_old", "tag_old_new", "tag_new_old"]   # điều kiện có ở notebook 05
TEMPLATES       = ["D1_tech_tactic", "D2_subtech_parent", "D4_id_currency"]
N_PASSAGES      = 4
MAX_CHUNK_WORDS = 250
MAX_MODEL_LEN   = 6144
MAX_TOKENS      = 160
GPU_MEM         = 0.85
SEED            = 2026
BOOT_B          = 2000
SMOKE_PER_GROUP = 1
MECH_VERSION    = "1.0"

REQUIRED = {
    "pilot_queries":  DRIFT / "pilot_queries.jsonl",
    "pilot_items":    DRIFT / "pilot_items.jsonl",
    "drift_manifest": DRIFT / "manifest.json",
    "tactic_alias":   DRIFT / "tactic_alias.json",
    "corpus_old":     ROOT / "benchmark" / f"attack-{V_OLD}" / "corpus.jsonl",
    "corpus_new":     ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
    "scoring_module": ROOT / "ctidrift_scoring.py",
    "infer_module":   ROOT / "ctidrift_infer.py",
}
missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, "Thiếu đầu vào — hãy chạy notebook 02–05 trước:\n  " + "\n  ".join(missing)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

DM = json.loads(REQUIRED["drift_manifest"].read_text())
for k in ("pilot_queries", "pilot_items"):
    assert DM["outputs"][f"{k}.jsonl"] == sha256(REQUIRED[k]), f"{k}.jsonl không khớp manifest notebook 03"
assert "structured_outputs_config" in REQUIRED["infer_module"].read_text(), \
    "ctidrift_infer.py là bản cũ — hãy chạy lại notebook 04 bản 1.1 (ô %%writefile) trước"

ENV = {**os.environ, "HF_HOME": HF_HOME, "HF_HUB_CACHE": f"{HF_HOME}/hub", "VLLM_CACHE_ROOT": VLLM_CACHE_ROOT,
       "VLLM_USE_FLASHINFER_SAMPLER": "0", "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("HF_HUB_OFFLINE", None)

MANIFEST = {"mech_version": MECH_VERSION, "v_old": V_OLD, "v_new": V_NEW,
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(CONDITIONS=CONDITIONS, TEMPLATES=TEMPLATES, N_PASSAGES=N_PASSAGES,
                           MAX_CHUNK_WORDS=MAX_CHUNK_WORDS, MAX_MODEL_LEN=MAX_MODEL_LEN, MAX_TOKENS=MAX_TOKENS,
                           GPU_MEM=GPU_MEM, SEED=SEED, BOOT_B=BOOT_B),
            "models": MODELS,
            "inputs": {k: {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)} for k, p in REQUIRED.items()}}
for k, v in MANIFEST["inputs"].items():
    print(f"{k:15s} {v['sha256'][:12]}…  {v['path']}")
print("Kết quả notebook 04:", "có" if (PILOT / "scored.csv").exists() else "KHÔNG có (bỏ qua phần chấm lại 04)")
print("Kết quả notebook 05:", "có" if (CTX / "scored.csv").exists() else "KHÔNG có (bỏ qua phần chấm lại 05)")
print("Prompt notebook 05: ", "có" if (CTX / "prompts").exists() else "KHÔNG có (bỏ qua kiểm tra trùng prompt)")

pilot_queries   1382ad00e2b4…  benchmark/drift-18.1-19.2/pilot_queries.jsonl
pilot_items     d76133e1cd33…  benchmark/drift-18.1-19.2/pilot_items.jsonl
drift_manifest  6abfab5da091…  benchmark/drift-18.1-19.2/manifest.json
tactic_alias    a5efc7448153…  benchmark/drift-18.1-19.2/tactic_alias.json
corpus_old      d435b77a7e8f…  benchmark/attack-18.1/corpus.jsonl
corpus_new      0198ec96b46b…  benchmark/attack-19.2/corpus.jsonl
scoring_module  56b3cb0eb0c8…  ctidrift_scoring.py
infer_module    833c7fe70aba…  ctidrift_infer.py
Kết quả notebook 04: có
Kết quả notebook 05: có
Prompt notebook 05:  có


## 2. Phần A — Bộ chấm v1.3

Ô dưới ghi file `ctidrift_scoring_v13.py`. File này **gọi lại** `score_drift` của v1.2 (không sửa v1.2) rồi bổ sung:

- **`tactic_pairs`**: tách các cặp (mã, tên) trong câu trả lời JSON. Hỗ trợ các dạng mô hình hay viết:
  - `"TA0005 Stealth"`;
  - dạng tách `["TA0005", "Stealth"]`;
  - `"Stealth (TA0005)"`;
  - nhiều mã gộp trong một mục, như `"TA0002, TA0005"`.

  Dạng tên đứng trước mã trong hai mục rời (`["Stealth", "TA0005"]`) không được ghép cặp. Dạng này không xuất hiện trong dữ liệu notebook 04 và 05.
- **`inconsistent`**: có ít nhất một cặp mà tên **không** phải tên của mã đó ở **cả hai** phiên bản, hoặc có mã tactic không tồn tại.
  - Tên được so theo cụm từ, nên `"Tactics Execution"` vẫn khớp `Execution`.
  - Khi so, bỏ qua khác biệt hoa thường, `&` được coi là `and`, và "Defence" được coi là "Defense". Gạch nối vẫn là một phần của từ, như v1.2: "Pre-Execution" **không** khớp Execution.
  - Tên cũ viết theo chính tả Anh ("Defence Evasion") vẫn được tính là tên cũ.
- **`outcome13`**, xét lần lượt theo thứ tự ưu tiên:
  1. `NO_ANSWER`: không có câu trả lời.
  2. `INCONSISTENT`: mã–tên không nhất quán.
  3. `STALE`: lỗi thời theo mã (`drift_ext` như notebook 04/05) **hoặc** có tên chỉ tồn tại ở bản cũ (`label_drift`).
  4. `CORRECT`: khớp đúng đáp án.
  5. `PARTIAL`: đúng một phần.
  6. `WRONG`: còn lại.
- **`stale13`** = (`outcome13 == "STALE"`): chỉ số lỗi thời theo cách chấm chặt.

Quy tắc này khớp với hướng dẫn chấm tay bản 2. Trường hợp như "TA0005 Defense Evasion" trên một kỹ thuật chỉ có ở v19.2 được tính là `STALE`, vì "Defense Evasion" là tên chỉ còn ở v18.1.

In [2]:
%%writefile ctidrift_scoring_v13.py
# Bộ chấm v1.3 (notebook 06). Gọi lại score_drift của v1.2, bổ sung kiểm tra mã–tên và lỗi thời theo tên.
from __future__ import annotations
import re
from ctidrift_scoring import score_drift, json_answer

SCORER_VERSION = "1.3"
_TA_ITEM = re.compile(r"^\s*(TA\d{4})\b\s*[:\-–]?\s*(.*?)\s*$", re.I)
_NAME_TA = re.compile(r"^\s*(.*?)\s*\(\s*(TA\d{4})\s*\)\s*$", re.I)      # "Impact (TA0040)"
_TA_START = re.compile(r"^\s*TA\d{4}\b", re.I)

def _split_items(items):
    # Tách mục gộp: "TA0005, TA0112", "TA0005/TA0112", "TA0005 Stealth, TA0112 Impact".
    # Trả về (phần, mục_gốc_có_bị_tách_không) để chỉ ghép "mã trần + mục kế tiếp" giữa các mục KHÔNG bị tách.
    out = []
    for x in items:
        parts = []
        for part in re.split(r"[,;/]", str(x)):
            parts += [p.strip() for p in re.split(r"(?<!\()(?<!\( )(?=\bTA\d{4}\b)", part, flags=re.I) if p.strip()]
        out += [(p, len(parts) > 1) for p in parts]
    return out

def tactic_pairs(answer: str) -> list:
    # Trả về danh sách (mã TA, tên đi kèm hoặc ""). Các dạng: "TA0005 Stealth", ["TA0005", "Stealth"],
    # "Stealth (TA0005)". Dạng tên đứng trước mã trong hai mục rời (["Stealth", "TA0005"]) không được ghép.
    items = json_answer(answer or "")
    if not items:
        return []
    toks = _split_items(items)
    items = [t for t, _ in toks]
    out, i = [], 0
    while i < len(items):
        m = _TA_ITEM.match(items[i])
        if not m:
            mn = _NAME_TA.match(items[i])
            if mn:
                out.append((mn.group(2).upper(), mn.group(1).strip(" ()")))
            i += 1
            continue
        tid, name = m.group(1).upper(), m.group(2).strip(" ()")
        if name:
            out.append((tid, name)); i += 1
        elif (i + 1 < len(items) and not _TA_START.match(items[i + 1])
              and not toks[i][1] and not toks[i + 1][1]):
            out.append((tid, items[i + 1].strip(" ()"))); i += 2
        else:
            out.append((tid, "")); i += 1
    return out

def _norm(s: str) -> str:
    # So tên: không phân biệt hoa thường, "&" = "and", "Defence" = "Defense"; gạch nối vẫn là một phần của từ (như v1.2)
    s = s.lower().replace("&", " and ").replace("defence", "defense")
    return re.sub(r"\s+", " ", s).strip()

def _names_of(tactic_alias: dict) -> dict:
    by_id = {}
    for n, t in tactic_alias.items():
        by_id.setdefault(t, set()).add(_norm(n))
    return by_id

def bad_pairs(answer: str, tactic_alias: dict) -> list:
    # Cặp mã–tên không nhất quán (tên không thuộc mã ở phiên bản nào) hoặc mã không tồn tại
    by_id, out = _names_of(tactic_alias), []
    for tid, name in tactic_pairs(answer):
        valid = by_id.get(tid)
        if valid is None:
            out.append((tid, name)); continue
        n = _norm(name)
        if n and not any(re.search(r"(?<![\w-])" + re.escape(v) + r"(?![\w-])", n) for v in valid):
            out.append((tid, name))
    return out

def score_v13(q: dict, answer: str, tactic_alias: dict) -> dict:
    s = score_drift(q, answer, tactic_alias)
    pred, gold, other = set(s["pred"]), set(q["gold"]), set(q["other_gold"])
    other_exact = bool(pred) and pred == other and other != gold
    drift_ext = s["outcome"] in ("STALE", "ANACHRONISM") or other_exact
    bad = bad_pairs(answer, tactic_alias) if q["answer_type"] == "tactic_set" else []
    # Tên cũ viết khác chính tả ("Defence Evasion") mà v1.2 không bắt được
    old = [_norm(x) for x in q.get("other_only_labels", [])]
    name_stale = q["answer_type"] == "tactic_set" and any(
        re.search(r"(?<![\w-])" + re.escape(o) + r"(?![\w-])", _norm(n)) for _, n in tactic_pairs(answer) for o in old)
    if s["outcome"] == "NO_ANSWER":
        o13 = "NO_ANSWER"
    elif bad:
        o13 = "INCONSISTENT"
    elif drift_ext or s["label_drift"] or name_stale:
        o13 = "STALE"
    elif s["em"]:
        o13 = "CORRECT"
    elif s["outcome"] == "PARTIAL":
        o13 = "PARTIAL"
    else:
        o13 = "WRONG"
    return {**s, "other_exact": other_exact, "drift_ext": drift_ext, "inconsistent": bool(bad),
            "bad_pairs": bad, "outcome13": o13, "stale13": o13 == "STALE", "scorer": SCORER_VERSION}

Writing ctidrift_scoring_v13.py


In [3]:
import importlib, ctidrift_scoring, ctidrift_scoring_v13
importlib.reload(ctidrift_scoring); importlib.reload(ctidrift_scoring_v13)
from ctidrift_scoring import score_drift
from ctidrift_scoring_v13 import score_v13, tactic_pairs

ALIAS_FILE = json.loads(REQUIRED["tactic_alias"].read_text())
TACTIC_ALIAS = ALIAS_FILE["alias"]

# Kiểm thử đơn vị: các ca lấy từ đợt chấm tay pilot 04
def _q(gold, other, olabels=("Defense Evasion",)):
    return dict(answer_type="tactic_set", gold=gold, other_gold=other, direction="current",
                other_only_labels=list(olabels), entity_names=[])
J = lambda *xs: json.dumps({"answer": list(xs)})
CASES = [
    # (mô tả, q, câu trả lời, outcome13 mong đợi)
    ("đúng, tên mới",                 _q(["TA0005"], ["TA0005"]), J("TA0005 Stealth"), "CORRECT"),
    ("đúng mã, tên cũ",               _q(["TA0005"], ["TA0005"]), J("TA0005 Defense Evasion"), "STALE"),
    ("dạng tách, lẫn tên cũ",          _q(["TA0005"], ["TA0005"]), J("TA0005", "Stealth", "TA0005", "Defense Evasion"), "STALE"),
    ("dạng tách, đúng",               _q(["TA0005"], ["TA0005"]), J("TA0005", "Stealth"), "CORRECT"),
    ("mã–tên lệch",                    _q(["TA0005"], ["TA0005"]), J("TA0005", "Impact"), "INCONSISTENT"),
    ("lệch, mã chỉ đúng bản cũ",        _q(["TA0112"], ["TA0005"]), J("TA0005 Privilege Escalation"), "INCONSISTENT"),
    ("ghép lai mã cũ + tên mới",        _q(["TA0112"], ["TA0005"]), J("TA0005 Defense Impairment"), "INCONSISTENT"),
    ("mã không tồn tại",              _q(["TA0112"], ["TA0005"]), J("TA0112 Defense Impairment", "TA0118 Defense Evasion"), "INCONSISTENT"),
    ("chữ thừa nhỏ vẫn khớp",          _q(["TA0002", "TA0005"], ["TA0003", "TA0004", "TA0005"]),
                                      J("TA0002 Tactics Execution", "TA0005 Stealth"), "CORRECT"),
    ("lỗi thời theo mã",               _q(["TA0112"], ["TA0005"]), J("TA0005 Defense Evasion"), "STALE"),
    ("đủ mới nhưng lẫn cũ",             _q(["TA0112"], ["TA0005"]), J("TA0005 Defense Evasion", "TA0112 Defense Impairment"), "STALE"),
    ("kỹ thuật mới, tên cũ (stt 109)",  _q(["TA0112"], []), J("TA0005 Defense Evasion", "TA0112 Defense Impairment"), "STALE"),
    ("rỗng",                           _q(["TA0112"], ["TA0005"]), J(), "NO_ANSWER"),
    ("sai hẳn",                        _q(["TA0042"], ["TA0042"]), J("TA0001 Initial Access"), "WRONG"),
    ("thiếu một mục",                  _q(["TA0003", "TA0005"], ["TA0003", "TA0005"]), J("TA0003 Persistence"), "PARTIAL"),
    ("dấu & và khoảng trắng thừa",       _q(["TA0011"], ["TA0011"]), J("TA0011 Command  &  Control"), "CORRECT"),
    ("tên cũ viết kiểu Anh",            _q(["TA0005"], ["TA0005"]), J("TA0005 Defence Evasion"), "STALE"),
    ("từ ghép có gạch nối không khớp",   _q(["TA0002"], ["TA0002"]), J("TA0002 Pre-Execution"), "INCONSISTENT"),
    ("tên trước mã trong một mục",       _q(["TA0002", "TA0005"], ["TA0002", "TA0005"]),
                                      J("Stealth, TA0005", "Execution, TA0002"), "CORRECT"),
    ("hai mã trong một mục",            _q(["TA0002", "TA0005"], ["TA0002", "TA0005"]), J("TA0002, TA0005"), "CORRECT"),
    ("cặp thứ hai trong mục gộp lệch",   _q(["TA0005"], ["TA0005"]), J("TA0005 Stealth, TA0112 Impact"), "INCONSISTENT"),
    ("tên trước mã, lệch",              _q(["TA0005"], ["TA0005"]), J("Impact (TA0005)"), "INCONSISTENT"),
    ("tên trước mã, đúng",              _q(["TA0005"], ["TA0005"]), J("Stealth (TA0005)"), "CORRECT"),
]
bad = [(d, score_v13(q, a, TACTIC_ALIAS)["outcome13"], exp) for d, q, a, exp in CASES
       if score_v13(q, a, TACTIC_ALIAS)["outcome13"] != exp]
assert not bad, f"Bộ chấm v1.3 sai ở: {bad}"
# Câu hỏi kỹ thuật (D2/D4) không bị ảnh hưởng bởi kiểm tra mã–tên
qt = dict(answer_type="technique_set", gold=["T1686"], other_gold=["T1562.004"], direction="current",
          other_only_labels=["Defense Evasion"], entity_names=[])
assert score_v13(qt, J("T1562.004"), TACTIC_ALIAS)["outcome13"] == "STALE"
assert score_v13(qt, J("T1686"), TACTIC_ALIAS)["outcome13"] == "CORRECT"
print(f"✓ Bộ chấm v1.3 qua {len(CASES) + 2} ca kiểm thử")

✓ Bộ chấm v1.3 qua 25 ca kiểm thử


## 3. Phần B — Chấm lại notebook 04 và 05 bằng v1.3 (không cần GPU)

Ta lấy đúng các câu trả lời đã lưu trong `scored.csv` của hai notebook và chấm lại. Mỗi bảng đặt cạnh nhau:

- **v1.2**: `drift_ext`, chỉ xét mã. Đây là chỉ số chính đã báo cáo.
- **v1.3**: `stale13`, lỗi thời theo mã hoặc tên, và mã–tên phải nhất quán.
- tỉ lệ `INCONSISTENT`;
- lỗi thời v1.3 **chỉ tính trên các câu nhất quán mã–tên**.

**Cách đọc:**

- `stale13` thấp hơn `drift_ext` có thể chỉ vì nhiều câu trả lời bị xếp `INCONSISTENT` (cặp mã–tên vô nghĩa), chứ không phải vì mô hình bớt lỗi thời. Vì vậy luôn đọc `stale13` cùng cột `INCONSISTENT`.
- Nếu các cột cho cùng một bức tranh thì kết luận của notebook 04 và 05 **không phụ thuộc** vào cách chấm.

**Phạm vi dữ liệu:**

- Notebook 04: lấy mọi câu hỏi về bản hiện hành, cả hai cách hỏi (`current` và `unversioned`), giống bảng chính của notebook 04.
- Notebook 05: lấy cách hỏi `current`, giống phần kiểm định của notebook 05.

In [4]:
QUERIES_ALL = [json.loads(l) for l in open(REQUIRED["pilot_queries"], encoding="utf-8")]
QMAP = {q["query_id"]: q for q in QUERIES_ALL}

def rescore(df):
    rows = []
    for r in df.itertuples(index=False):
        s = score_v13(QMAP[r.query_id], r.answer if isinstance(r.answer, str) else "", TACTIC_ALIAS)
        rows.append(dict(drift_ext13=s["drift_ext"], stale13=s["stale13"], inconsistent=s["inconsistent"],
                         outcome13=s["outcome13"], em13=s["em"]))
    out = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    for c in ("drift_ext", "em", "label_drift", "revoked"):     # CSV có thể đọc ra chuỗi "True"/"False"
        if c in out:
            out[c] = out[c].astype(str).str.strip().str.lower().eq("true")
    # Kiểm tra: phần v1.2 tính lại phải trùng với cột đã lưu
    assert (out["drift_ext"] == out["drift_ext13"]).all(), "drift_ext tính lại không khớp — gửi thầy"
    assert (out["em"] == out["em13"]).all(), "em tính lại không khớp — gửi thầy"
    return out

def compare_table(df, conds, group_mask_changed, group_mask_label):
    rows = []
    for (m, c), g in df.groupby(["model", "condition"]):
        if c not in conds:
            continue
        ch, lb = g[group_mask_changed(g)], g[group_mask_label(g)]
        r3 = lambda x: round(x, 3) if x == x else None
        cons = ch[~ch.inconsistent]
        rows.append({"model": m, "condition": c, "n changed": len(ch),
                     "changed: lỗi thời v1.2": r3(ch.drift_ext.mean()),
                     "changed: lỗi thời v1.3": r3(ch.stale13.mean()),
                     "changed: INCONSISTENT": r3(ch.inconsistent.mean()),
                     "changed: lỗi thời v1.3 / câu nhất quán": r3(cons.stale13.mean()) if len(cons) else None,
                     "n label": len(lb),
                     "label: tên cũ v1.2": r3(lb.label_drift.mean()) if len(lb) else None,
                     "label: lỗi thời v1.3": r3(lb.stale13.mean()) if len(lb) else None,
                     "label: INCONSISTENT": r3(lb.inconsistent.mean()) if len(lb) else None})
    t = pd.DataFrame(rows)
    t["condition"] = pd.Categorical(t["condition"], conds)
    return t.sort_values(["model", "condition"]).reset_index(drop=True)

RESCORED = {}
if (PILOT / "scored.csv").exists():
    S4 = rescore(pd.read_csv(PILOT / "scored.csv"))
    S4 = S4[S4.direction == "current"]          # như bảng chính của notebook 04: cả cách hỏi current và unversioned
    RESCORED["04"] = S4
    T_R4 = compare_table(S4, ["no_rag", "rag_new", "rag_mixed"],
                         lambda g: g.query_id.map(lambda i: QMAP[i]["answer_changed"]).astype(bool),
                         lambda g: g.pair_class == "LABEL_CHANGED")
    T_R4.to_csv(OUT / "rescore_04.csv", index=False)
    print("Notebook 04 (hỏi về bản hiện hành, cả hai cách hỏi):"); display(T_R4)
if (CTX / "scored.csv").exists():
    S5 = rescore(pd.read_csv(CTX / "scored.csv"))
    RESCORED["05"] = S5
    T_R5 = compare_table(S5[S5.framing == "current"], ["new", "old", "old_new", "new_old", "tag_old_new",
                                                       "tag_new_old", "old_changelog"],
                         lambda g: g.group == "changed", lambda g: g.group == "label")
    T_R5.to_csv(OUT / "rescore_05.csv", index=False)
    print("Notebook 05 (cách hỏi current):"); display(T_R5)

Notebook 04 (hỏi về bản hiện hành, cả hai cách hỏi):


,model,condition,n changed,changed: lỗi thời v1.2,changed: lỗi thời v1.3,changed: INCONSISTENT,changed: lỗi thời v1.3 / câu nhất quán,n label,label: tên cũ v1.2,label: lỗi thời v1.3,label: INCONSISTENT
0,llama3.1-8b,no_rag,224,0.598,0.188,0.674,0.575,160,0.200,0.081,0.888
1,llama3.1-8b,rag_new,224,0.107,0.071,0.174,0.086,160,0.006,0.000,0.019
2,llama3.1-8b,rag_mixed,224,0.906,0.938,0.004,0.942,160,0.862,0.831,0.031
3,qwen3-8b,no_rag,224,0.223,0.179,0.397,0.296,160,0.031,0.019,0.431
4,qwen3-8b,rag_new,224,0.045,0.040,0.031,0.041,160,0.000,0.000,0.025
5,qwen3-8b,rag_mixed,224,0.826,0.862,0.009,0.869,160,0.931,0.931,0.012


Notebook 05 (cách hỏi current):


,model,condition,n changed,changed: lỗi thời v1.2,changed: lỗi thời v1.3,changed: INCONSISTENT,changed: lỗi thời v1.3 / câu nhất quán,n label,label: tên cũ v1.2,label: lỗi thời v1.3,label: INCONSISTENT
0,llama3.1-8b,new,112,0.098,0.080,0.214,0.102,80,0.000,0.000,0.000
1,llama3.1-8b,old,112,1.000,1.000,0.000,1.000,80,1.000,1.000,0.000
2,llama3.1-8b,old_new,112,0.929,0.955,0.018,0.973,80,0.962,0.950,0.012
3,llama3.1-8b,new_old,112,0.777,0.768,0.018,0.782,80,0.912,0.912,0.000
4,llama3.1-8b,tag_old_new,112,0.054,0.054,0.000,0.054,80,0.088,0.088,0.000
5,llama3.1-8b,tag_new_old,112,0.223,0.223,0.009,0.225,80,0.362,0.362,0.000
6,llama3.1-8b,old_changelog,112,0.223,0.214,0.009,0.216,80,0.000,0.000,0.000
7,qwen3-8b,new,112,0.018,0.018,0.080,0.019,80,0.000,0.000,0.000
8,qwen3-8b,old,112,1.000,1.000,0.000,1.000,80,1.000,1.000,0.000
9,qwen3-8b,old_new,112,0.938,0.893,0.054,0.943,80,0.975,0.975,0.000


Giả thuyết H1–H4 của notebook 05, tính lại với chỉ số chặt `stale13`. Khoảng tin cậy dùng **bootstrap theo cụm dòng dõi**, giống hệt notebook 05.

In [5]:
CLUSTER = "lineage"

def _agg(df, col):
    g = df.groupby(CLUSTER)[col].agg(["sum", "count"])
    return g["sum"].to_numpy(float), g["count"].to_numpy(float), g.index

def boot_rate(df, col):
    if len(df) == 0:
        return (float("nan"),) * 3
    s, c, _ = _agg(df, col)
    idx = np.random.default_rng(SEED).integers(0, len(s), size=(BOOT_B, len(s)))
    r = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (s.sum() / c.sum(), lo, hi)

def boot_diff_paired(df_a, df_b, col):
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib), "hai nhóm phải có cùng tập mục"
    if len(sa) == 0:
        return (float("nan"),) * 3
    idx = np.random.default_rng(SEED).integers(0, len(sa), size=(BOOT_B, len(sa)))
    r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi)

def fmt_b(t):
    v, lo, hi = t
    return "—" if v != v else f"{v:.2f} [{lo:.2f}, {hi:.2f}]"

def verdict(t):
    return "✓ (>0)" if t[1] > 0 else ("ngược chiều (<0)" if t[2] < 0 else "·")

if "05" in RESCORED:
    S5c = RESCORED["05"][RESCORED["05"].framing == "current"]
    H4_MASK = ~((S5c.template == "D1_tech_tactic") & S5c.revoked)
    HYP05 = [("H1 vị trí", "old_new", "new_old", False), ("H2 có mặt đoạn cũ", "new_old", "new", False),
             ("H3a nhãn (cũ trước)", "old_new", "tag_old_new", False),
             ("H3b nhãn (mới trước)", "new_old", "tag_new_old", False),
             ("H4 ghi chú thay đổi", "old", "old_changelog", True)]
    rows = []
    for m in [x["tag"] for x in MODELS]:
        for name, a, b, h4 in HYP05:
            d = S5c[(S5c.model == m) & (S5c.group == "changed") & (H4_MASK if h4 else True)]
            A, B = d[d.condition == a].sort_values("query_id"), d[d.condition == b].sort_values("query_id")
            v12, v13 = boot_diff_paired(A, B, "drift_ext"), boot_diff_paired(A, B, "stale13")
            rows.append({"model": m, "giả thuyết": name, "v1.2 (drift_ext)": fmt_b(v12), "hỗ trợ v1.2": verdict(v12),
                         "v1.3 (stale13)": fmt_b(v13), "hỗ trợ v1.3": verdict(v13)})
    T_H05 = pd.DataFrame(rows)
    T_H05.to_csv(OUT / "rescore_05_hypotheses.csv", index=False)
    display(T_H05)

,model,giả thuyết,v1.2 (drift_ext),hỗ trợ v1.2,v1.3 (stale13),hỗ trợ v1.3
0,qwen3-8b,H1 vị trí,"0.10 [0.03, 0.17]",✓ (>0),"0.02 [-0.07, 0.11]",·
1,qwen3-8b,H2 có mặt đoạn cũ,"0.82 [0.75, 0.89]",✓ (>0),"0.86 [0.79, 0.92]",✓ (>0)
2,qwen3-8b,H3a nhãn (cũ trước),"0.86 [0.78, 0.93]",✓ (>0),"0.81 [0.73, 0.89]",✓ (>0)
3,qwen3-8b,H3b nhãn (mới trước),"0.57 [0.46, 0.69]",✓ (>0),"0.61 [0.49, 0.72]",✓ (>0)
4,qwen3-8b,H4 ghi chú thay đổi,"1.00 [1.00, 1.00]",✓ (>0),"1.00 [1.00, 1.00]",✓ (>0)
5,llama3.1-8b,H1 vị trí,"0.15 [0.08, 0.23]",✓ (>0),"0.19 [0.09, 0.29]",✓ (>0)
6,llama3.1-8b,H2 có mặt đoạn cũ,"0.68 [0.57, 0.78]",✓ (>0),"0.69 [0.59, 0.78]",✓ (>0)
7,llama3.1-8b,H3a nhãn (cũ trước),"0.88 [0.81, 0.93]",✓ (>0),"0.90 [0.84, 0.96]",✓ (>0)
8,llama3.1-8b,H3b nhãn (mới trước),"0.55 [0.45, 0.66]",✓ (>0),"0.54 [0.44, 0.65]",✓ (>0)
9,llama3.1-8b,H4 ghi chú thay đổi,"0.90 [0.84, 0.96]",✓ (>0),"0.90 [0.84, 0.96]",✓ (>0)


## 4. Phần C — Truy vấn, bằng chứng và hai cách hỏi

Ta dựng lại **đúng** tập truy vấn và bằng chứng của notebook 05 (cách hỏi `current`, mẫu D1, D2, D4; ba nhóm `changed`, `label`, `stable`), rồi thêm biến thể `nover` cho từng câu. Có ba dạng câu hỏi, và cách bỏ số phiên bản như sau:

| Mẫu | Câu `ver` | Câu `nover` |
|---|---|---|
| D1 | According to the current MITRE ATT&CK Enterprise release **(v19.2)**, which tactic(s)… | According to the current MITRE ATT&CK Enterprise release, which tactic(s)… |
| D2 | In the current MITRE ATT&CK Enterprise release **(v19.2)**, what is the parent… | In the current MITRE ATT&CK Enterprise release, what is the parent… |
| D4 | **As of MITRE ATT&CK Enterprise v19.2**, is the technique ID… | **In the current MITRE ATT&CK Enterprise release**, is the technique ID… |

Ô dưới tự kiểm tra: mọi câu `nover` không còn số phiên bản nào, vẫn có chữ *current*, và mọi câu `ver` đổi được.

In [6]:
ITEMS = {it["item_id"]: it for it in map(json.loads, open(REQUIRED["pilot_items"], encoding="utf-8"))}
CORP = {V_OLD: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_old"], encoding="utf-8"))},
        V_NEW: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_new"], encoding="utf-8"))}}

def group_of(q):                         # giống notebook 05
    if q["answer_changed"]:
        return "changed"
    if q["pair_class"] == "LABEL_CHANGED":
        return "label"
    if q["pair_class"] == "STABLE":
        return "stable"
    return None

def evidence_new(q, it):                 # giống notebook 05
    ev = q["evidence"][V_NEW]
    return ev if ev else [f"{s}#0" for s in it["anchor_new"] if f"{s}#0" in CORP[V_NEW]]

def nover(question):
    out = question.replace(f" (v{V_NEW})", "")
    out = re.sub(rf"^As of MITRE ATT&CK Enterprise v{re.escape(V_NEW)},", "In the current MITRE ATT&CK Enterprise release,", out)
    return out

BASE, EV = [], {}
for q in QUERIES_ALL:
    g = group_of(q)
    if q["direction"] != "current" or q["framing"] != "current" or q["template"] not in TEMPLATES or g is None:
        continue
    it = ITEMS[q["item_id"]]
    eo, en = q["evidence"][V_OLD], evidence_new(q, it)
    assert eo and en, f"{q['query_id']}: thiếu bằng chứng O hoặc N"
    BASE.append({**q, "group": g})
    EV[q["query_id"]] = (eo, en)

QUERIES = []
for q in BASE:
    QUERIES.append({**q, "base_id": q["query_id"], "qform": "ver"})
    qn = nover(q["question"])
    assert qn != q["question"], f"{q['query_id']}: không bỏ được số phiên bản khỏi câu hỏi"
    assert V_NEW not in qn and V_OLD not in qn and not re.search(r"\bv\d", qn) and "current" in qn, \
        f"{q['query_id']}: câu nover sai: {qn}"
    QUERIES.append({**q, "question": qn, "base_id": q["query_id"], "query_id": q["query_id"] + "_nover",
                    "qform": "nover"})
QBYID = {q["query_id"]: q for q in QUERIES}
assert len({q["question"] for q in QUERIES}) == len(QUERIES), "có hai truy vấn trùng câu hỏi"
with open(OUT / "queries.jsonl", "w", encoding="utf-8") as f:
    for q in QUERIES:
        f.write(json.dumps(q, ensure_ascii=False, sort_keys=True) + "\n")
print(f"{len(BASE)} câu gốc × 2 cách hỏi = {len(QUERIES)} truy vấn:",
      dict(collections.Counter((q['template'][:2], q['group']) for q in BASE)))
for qid in ["D1-T1222|current", "D4-T1562.004|current"]:
    if qid in QBYID:
        print("\n ver  :", QBYID[qid]["question"][:150], "\n nover:", QBYID[qid + "_nover"]["question"][:150])

312 câu gốc × 2 cách hỏi = 624 truy vấn: {('D1', 'stable'): 40, ('D1', 'label'): 80, ('D1', 'changed'): 87, ('D2', 'stable'): 40, ('D2', 'changed'): 8, ('D4', 'stable'): 40, ('D4', 'changed'): 17}

 ver  : According to the current MITRE ATT&CK Enterprise release (v19.2), which tactic(s) does the technique File and Directory Permissions Modification (T122 
 nover: According to the current MITRE ATT&CK Enterprise release, which tactic(s) does the technique File and Directory Permissions Modification (T1222) belon

 ver  : As of MITRE ATT&CK Enterprise v19.2, is the technique ID T1562.004 (Disable or Modify System Firewall) still valid? If yes, repeat the ID; if it has b 
 nover: In the current MITRE ATT&CK Enterprise release, is the technique ID T1562.004 (Disable or Modify System Firewall) still valid? If yes, repeat the ID; 


## 5. Ghi chú cầu nối và đoạn nhiễu

- **Ghi chú cầu nối (B)** dùng đúng câu "revoked and replaced by" của notebook 05, và **chỉ** cho câu có mã bị thu hồi.
- **Đoạn nhiễu** được chọn y như notebook 05: BM25 trên kho v19.2 theo câu hỏi **`ver`**, loại mọi đoạn có mã hoặc tên tactic của đáp án hai phiên bản. Câu `nover` dùng **cùng** đoạn nhiễu với câu `ver` tương ứng, để hai cách hỏi chỉ khác nhau ở câu hỏi.

In [7]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class BM25:                            # giống hệt notebook 04, 05
    def __init__(self, docs, k1=1.2, b=0.75):
        self.ids = sorted(docs)
        n = len(self.ids)
        toks = [collections.Counter(tokenize(docs[i])) for i in self.ids]
        dl = np.array([sum(t.values()) for t in toks], dtype=float)
        avgdl = dl.mean()
        post = collections.defaultdict(lambda: ([], []))
        for j, tf in enumerate(toks):
            for t, c in tf.items():
                post[t][0].append(j)
                post[t][1].append(c)
        self.post = {}
        for t, (idx, tf) in post.items():
            idx, tf = np.array(idx), np.array(tf, dtype=float)
            idf = math.log(1 + (n - len(idx) + 0.5) / (len(idx) + 0.5))
            self.post[t] = (idx, idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[idx] / avgdl)))
        self.n = n

    def search(self, query, k=5):
        scores = np.zeros(self.n)
        for t in sorted(set(tokenize(query))):
            if t in self.post:
                idx, w = self.post[t]
                scores[idx] += w
        cand = np.nonzero(scores)[0]
        order = cand[np.lexsort((cand, -scores[cand]))][:k]
        return [(self.ids[j], float(scores[j])) for j in order]

INDEX_NEW = BM25({i: d["title"] + "\n" + d["text"] for i, d in CORP[V_NEW].items()})

_old_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["new_only_labels"]}
_new_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["old_only_labels"]}
OLD_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_old_names)}
NEW_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_new_names)}

def bridge_note(q):
    # Câu "revoked and replaced by" của notebook 05; None nếu mã còn hiệu lực
    it = ITEMS[q["item_id"]]
    if q["group"] != "changed" or it["anchor_status_new"] == "active":
        return None
    major = V_NEW.split(".")[0]
    names = it["name_new"].split(" / ")
    repl = ", ".join(f"{s} ({n})" for s, n in zip(it["anchor_new"], names))
    return (f"In MITRE ATT&CK v{major}, technique {it['anchor_old']} ({it['name_old']}) was revoked "
            f"and replaced by {repl}.")

def banned_terms(q):                   # giống notebook 05
    it = ITEMS[q["item_id"]]
    ids = {it["anchor_old"], *it["anchor_new"], *q["gold"], *q["other_gold"]}
    names = set()
    if q["answer_type"] == "tactic_set":
        for t in set(q["gold"]) | set(q["other_gold"]):
            names |= {OLD_NAME.get(t), NEW_NAME.get(t)} - {None}
    return ids, names

def distractors(q, k):                 # giống notebook 05 (q là câu gốc, cách hỏi ver)
    ids, names = banned_terms(q)
    id_re = re.compile(r"\b(?:" + "|".join(re.escape(b) for b in sorted(ids)) + r")(?![.\d])")
    name_re = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(names)) + r")\b", re.I) if names else None
    eo, en = EV[q["query_id"]]
    out = []
    for cid, _ in INDEX_NEW.search(q["question"], 3000):
        d = CORP[V_NEW][cid]
        if (cid in en or d["metadata"]["doc_id"] in ids or id_re.search(d["text"])
                or (name_re and name_re.search(d["text"]))):
            continue
        out.append(cid)
        if len(out) == k:
            break
    assert len(out) == k, f"{q['query_id']}: không đủ đoạn nhiễu"
    return out

BRIDGE   = {q["query_id"]: bridge_note(q) for q in BASE}
DISTRACT = {q["query_id"]: distractors(q, N_PASSAGES) for q in BASE}
leaks = []
for q in BASE:
    ids, names = banned_terms(q)
    for cid in DISTRACT[q["query_id"]]:
        t = CORP[V_NEW][cid]["text"]
        if any(re.search(re.escape(i) + r"(?![.\d])", t) for i in ids) or any(n.lower() in t.lower() for n in names):
            leaks.append((q["query_id"], cid))
assert not leaks, f"Đoạn nhiễu lộ đáp án: {leaks[:5]}"
n_rev = sum(1 for v in BRIDGE.values() if v)
print(f"✓ Đoạn nhiễu không lộ đáp án; {n_rev} câu gốc có ghi chú cầu nối (mã bị thu hồi)")

✓ Đoạn nhiễu không lộ đáp án; 40 câu gốc có ghi chú cầu nối (mã bị thu hồi)


## 6. Dựng prompt cho 10 điều kiện × 2 cách hỏi

Lời dặn hệ thống, yêu cầu theo kiểu đáp án và lược đồ JSON **giống hệt** notebook 04, 05.

Ô kiểm tra sau mục này xác nhận các điều sau:

1. Đúng thứ tự O/N/B và đúng 4 đoạn.
2. Đúng loại nhãn:
   - không nhãn ở `old_new`/`new_old`;
   - nhãn đầy đủ ở `tag_*`, `bridge_*`;
   - chỉ số phiên bản ở `vtag_*`;
   - nhãn tráo ở `swap_*`.
3. Hai cách hỏi có ngữ cảnh giống hệt nhau.
4. Bốn điều kiện lặp lại có prompt **giống hệt notebook 05** (nếu có thư mục prompt của notebook 05).

In [8]:
SYSTEM = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
          "Answer the user's question accurately. Respond only with a JSON object of the form {\"answer\": [...]}. "
          "Put only the requested items in the list. Use an empty list if you do not know.")
INSTR = {
    "tactic_set":     'List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".',
    "technique_set":  "List ATT&CK technique IDs only (format T1234 or T1234.001).",
    "mitigation_set": "List ATT&CK mitigation IDs only (format M1234).",
}
PATTERN = {"technique_set": "^T[0-9]{4}(\\.[0-9]{3})?$", "mitigation_set": "^M[0-9]{4}$",
           "tactic_set": "^[A-Za-z0-9 ,.()&/'-]{1,60}$"}
TAGS = {
    "full": {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]",
             V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]"},
    "ver":  {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD}]",
             V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW}]"},
}
SWAP_VER = {V_OLD: V_NEW, V_NEW: V_OLD}

def schema_for(answer_type):
    return {"type": "object",
            "properties": {"answer": {"type": "array", "items": {"type": "string", "pattern": PATTERN[answer_type]},
                                      "maxItems": 8}},
            "required": ["answer"], "additionalProperties": False}

def clip(text, n):
    words = list(re.finditer(r"\S+", text))
    return text if len(words) <= n else text[:words[n - 1].end()] + " …"

def passages(q, cond):
    # (loại, phiên bản nội dung, nhãn hiển thị hoặc None, văn bản)
    spec, bid = CONDITIONS[cond], q["base_id"]
    eo, en = EV[bid]
    out = []
    for kind in spec["order"]:
        if kind == "O":
            out += [("O", V_OLD, CORP[V_OLD][e]["text"]) for e in eo]
        elif kind == "N":
            out += [("N", V_NEW, CORP[V_NEW][e]["text"]) for e in en]
        elif kind == "B" and BRIDGE[bid]:
            out.append(("B", None, BRIDGE[bid]))
    out += [("D", V_NEW, CORP[V_NEW][d]["text"]) for d in DISTRACT[bid][:max(0, N_PASSAGES - len(out))]]
    res = []
    for k, v, t in out:
        label = None
        if spec["tag"] and v:
            shown = SWAP_VER[v] if (spec["swap"] and k in ("O", "N")) else v
            label = TAGS[spec["tag"]][shown]
        res.append((k, v, label, (label + "\n" if label else "") + clip(t, MAX_CHUNK_WORDS)))
    return res

def build_messages(q, cond):
    ctx = "\n\n".join(f"[{i + 1}] {t}" for i, (_, _, _, t) in enumerate(passages(q, cond)))
    user = ("Use the following context passages from a CTI knowledge base if they are relevant.\n\n"
            f"{ctx}\n\nQuestion: {q['question']}\n{INSTR[q['answer_type']]}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as f:
        for q in QUERIES:
            f.write(json.dumps(dict(query_id=q["query_id"], condition=cond, answer_type=q["answer_type"],
                                    messages=build_messages(q, cond), schema=schema_for(q["answer_type"]),
                                    layout=[k for k, *_ in passages(q, cond)]),
                               ensure_ascii=False, sort_keys=True) + "\n")
MANIFEST["prompts"] = {c: sha256(PROMPTS / f"{c}.jsonl") for c in CONDITIONS}
print(pd.Series({c: collections.Counter(tuple(k for k, *_ in passages(q, c)) for q in QUERIES).most_common(3)
                 for c in CONDITIONS}).to_string())

old_new                               [((O, N, D, D), 624)]
new_old                               [((N, O, D, D), 624)]
tag_old_new                           [((O, N, D, D), 624)]
tag_new_old                           [((N, O, D, D), 624)]
vtag_old_new                          [((O, N, D, D), 624)]
vtag_new_old                          [((N, O, D, D), 624)]
swap_old_new                          [((O, N, D, D), 624)]
swap_new_old                          [((N, O, D, D), 624)]
bridge_old_new    [((O, N, D, D), 544), ((O, N, B, D), 80)]
bridge_new_old    [((N, O, D, D), 544), ((N, O, B, D), 80)]


In [9]:
# Tự kiểm tra cấu trúc prompt
errs = []
for q in QUERIES:
    bid = q["base_id"]
    eo, en = EV[bid]
    for cond, spec in CONDITIONS.items():
        ps = passages(q, cond)
        exp = []
        for k in spec["order"]:
            exp += ["O"] * len(eo) if k == "O" else ["N"] * len(en) if k == "N" else (["B"] if BRIDGE[bid] else [])
        kinds = [k for k, *_ in ps]
        if kinds[:len(exp)] != exp or set(kinds[len(exp):]) - {"D"} or len(ps) != max(N_PASSAGES, len(exp)):
            errs.append((q["query_id"], cond, "bố cục"))
        for k, v, label, _ in ps:
            if k == "B":
                ok = label is None
            elif spec["tag"] is None:
                ok = label is None
            else:
                want_v = SWAP_VER[v] if (spec["swap"] and k in ("O", "N")) else v
                ok = label == TAGS[spec["tag"]][want_v]
            if not ok:
                errs.append((q["query_id"], cond, f"nhãn đoạn {k}"))
        text = build_messages(q, cond)[1]["content"]
        for style, tags in TAGS.items():
            present = any(t in text for t in tags.values())
            if present != (spec["tag"] == style):
                errs.append((q["query_id"], cond, f"nhãn kiểu {style}"))
    # Hai cách hỏi: cùng ngữ cảnh
    if q["qform"] == "nover":
        for cond in CONDITIONS:
            a = build_messages(q, cond)[1]["content"].split("\n\nQuestion:")[0]
            b = build_messages(QBYID[bid], cond)[1]["content"].split("\n\nQuestion:")[0]
            if a != b:
                errs.append((q["query_id"], cond, "ngữ cảnh ver/nover khác nhau"))
assert not errs, errs[:10]
print(f"✓ {len(QUERIES) * len(CONDITIONS)} prompt đúng cấu trúc")

# Kiểm tra trực tiếp quy tắc tráo nhãn (không dùng lại biểu thức của passages)
for q in QUERIES[:50]:
    for X in ("old_new", "new_old"):
        for k, v, label, t in passages(q, "swap_" + X):
            if k == "O":
                assert label == f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]", "tráo nhãn sai ở đoạn O"
            if k == "N":
                assert label == f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]", "tráo nhãn sai ở đoạn N"
print("✓ Tráo nhãn đúng: đoạn cũ mang nhãn v19.2 (current release), đoạn mới mang nhãn v18.1 (superseded)")

# Bốn điều kiện lặp lại phải trùng prompt notebook 05, và dùng cùng kho dữ liệu
if (CTX / "manifest.json").exists():
    m05 = json.loads((CTX / "manifest.json").read_text())["inputs"]
    diff = [k for k in ("pilot_queries", "pilot_items", "corpus_old", "corpus_new", "tactic_alias")
            if m05.get(k, {}).get("sha256") != MANIFEST["inputs"][k]["sha256"]]
    assert not diff, f"Dữ liệu đầu vào khác lần chạy notebook 05: {diff} — gửi thầy"
    print("✓ Dữ liệu đầu vào trùng với lần chạy notebook 05")
if (CTX / "prompts").exists():
    n_cmp = 0
    for cond in REPLICATED:
        old = {r["query_id"]: r for r in map(json.loads, open(CTX / "prompts" / f"{cond}.jsonl", encoding="utf-8"))}
        for q in QUERIES:
            if q["qform"] == "ver":
                assert old[q["query_id"]]["messages"] == build_messages(q, cond), \
                    f"Prompt {cond}/{q['query_id']} khác notebook 05 — gửi thầy"
                n_cmp += 1
    print(f"✓ {n_cmp} prompt của 4 điều kiện lặp lại giống hệt notebook 05")
else:
    print("(Không có prompt notebook 05 — bỏ qua kiểm tra trùng)")

ex = "D1-T1222|current"
if ex in QBYID:
    print("\nVí dụ swap_new_old, " + ex + ":\n" + build_messages(QBYID[ex], "swap_new_old")[1]["content"][:900])

✓ 6240 prompt đúng cấu trúc
✓ Tráo nhãn đúng: đoạn cũ mang nhãn v19.2 (current release), đoạn mới mang nhãn v18.1 (superseded)
✓ Dữ liệu đầu vào trùng với lần chạy notebook 05
✓ 1248 prompt của 4 điều kiện lặp lại giống hệt notebook 05

Ví dụ swap_new_old, D1-T1222|current:
Use the following context passages from a CTI knowledge base if they are relevant.

[1] [Source: MITRE ATT&CK Enterprise v18.1 (superseded)]
ATT&CK Technique T1222: File and Directory Permissions Modification
Tactics: Defense Impairment (TA0112).
Platforms: ESXi, Linux, macOS, Windows.
Adversaries may modify file or directory permissions/attributes to evade access control lists (ACLs) and access protected files. File and directory permissions are commonly managed by ACLs configured by the file or directory owner, or users with the appropriate permissions. File and directory ACL implementations vary by platform, but generally explicitly designate which users or groups can perform which actions (read, write, execute, 

In [10]:
smoke_ids = []
for key in sorted({(q["template"], q["group"], q["qform"]) for q in QUERIES}):
    smoke_ids += [q["query_id"] for q in QUERIES if (q["template"], q["group"], q["qform"]) == key][:SMOKE_PER_GROUP]
smoke_set = set(smoke_ids)
for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", encoding="utf-8") as fi, open(SMOKE_PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as fo:
        for line in fi:
            if json.loads(line)["query_id"] in smoke_set:
                fo.write(line)
print(f"Bộ chạy thử: {len(smoke_ids)} truy vấn/điều kiện")

Bộ chạy thử: 14 truy vấn/điều kiện


## 7. Chạy suy luận

Như notebook 05: mỗi mô hình chạy trong một tiến trình riêng bằng `ctidrift_infer.py`, và dừng ô thì tắt hẳn tiến trình GPU. Chạy thử trước, rồi chạy toàn bộ: 624 truy vấn × 10 điều kiện × 2 mô hình. Thời gian ước tính khoảng 25 phút mỗi mô hình trên RTX 4070.

**Theo dõi tiến độ:** ô chạy in ra lệnh `tail -f …` để em xem tiến độ trong terminal. Ô sẽ im lặng trong lúc chạy; đó là bình thường.

In [11]:
def run_infer(model, out_dir, prompts_dir, smoke=False):
    log = LOGS / f"{model['tag']}{'_smoke' if smoke else ''}.log"
    cmd = [sys.executable, "ctidrift_infer.py", "--repo", model["repo"], "--revision", model["revision"],
           "--tag", model["tag"], "--prompts-dir", str(prompts_dir), "--conditions", ",".join(CONDITIONS),
           "--out-dir", str(out_dir), "--max-model-len", str(MAX_MODEL_LEN), "--max-tokens", str(MAX_TOKENS),
           "--gpu-mem", str(GPU_MEM), "--seed", str(SEED),
           "--chat-template-kwargs", json.dumps(model["chat_template_kwargs"])]
    print(f"▶ {model['tag']}: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:\n    tail -f {log}", flush=True)
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV, cwd=ROOT, start_new_session=True)
        try:
            p.wait()
        except BaseException:
            def _kill(sig):
                try:
                    os.killpg(p.pid, sig)
                except ProcessLookupError:
                    pass
            _kill(signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except BaseException:
                pass
            _kill(signal.SIGKILL)
            print("⏹ Đã dừng và giải phóng GPU. Kiểm tra lại bằng lệnh nvidia-smi nếu cần.")
            raise
    tail = log.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(l for l in tail if l.startswith(f"[{model['tag']}]") or "LỖI" in l or "Error" in l)[-3000:])
    print(f"◀ {model['tag']}: {'XONG' if p.returncode == 0 else 'LỖI'} sau {(time.time() - t0) / 60:.1f} phút")
    if p.returncode != 0:
        print("\n".join(tail[-40:]))
        raise RuntimeError(f"Suy luận lỗi — gửi thầy file {log}")

SMOKE = OUT / "raw_smoke"
for m in MODELS:
    run_infer(m, SMOKE, SMOKE_PROMPTS, smoke=True)
fmt_bad = []
for m in MODELS:
    for cond in CONDITIONS:
        for r in map(json.loads, open(SMOKE / m["tag"] / f"{cond}.jsonl", encoding="utf-8")):
            if score_drift(QBYID[r["query_id"]], r["text"], TACTIC_ALIAS)["format"] != "json":
                fmt_bad.append((m["tag"], cond, r["query_id"], r["text"][:80]))
assert not fmt_bad, f"Có câu trả lời không phải JSON — gửi thầy: {fmt_bad[:5]}"
print("✓ Chạy thử: mọi câu trả lời đúng định dạng JSON")

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/mech-18.1-19.2/logs/qwen3-8b_smoke.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] old_new: 14 prompt, dài nhất 1445 token (giới hạn 5984)
[qwen3-8b] new_old: 14 prompt, dài nhất 1445 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] vtag_old_new: 14 prompt, dài nhất 1505 token (giới hạn 5984)
[qwen3-8b] vtag_new_old: 14 prompt, dài nhất 1505 token (giới hạn 5984)
[qwen3-8b] swap_old_new: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] swap_new_old: 14 prompt, dài nhất 1518 token (giới hạn 5984)
[qwen3-8b] bridge_old_new: 14 prompt, dài nhất 1517 token (giới hạn 5984)
[qwen3-8b] bridge_new_old: 14 prompt, dài nhất 1517 token (giới hạn 5984)
[qwen3-8b] old_new: xong 14 truy vấ

In [12]:
for m in MODELS:
    run_infer(m, RAW, PROMPTS)

▶ qwen3-8b: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/mech-18.1-19.2/logs/qwen3-8b.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] old_new: 624 prompt, dài nhất 1766 token (giới hạn 5984)
[qwen3-8b] new_old: 624 prompt, dài nhất 1766 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] vtag_old_new: 624 prompt, dài nhất 1826 token (giới hạn 5984)
[qwen3-8b] vtag_new_old: 624 prompt, dài nhất 1826 token (giới hạn 5984)
[qwen3-8b] swap_old_new: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] swap_new_old: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] bridge_old_new: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] bridge_new_old: 624 prompt, dài nhất 1839 token (giới hạn 5984)
[qwen3-8b] old_new: xong 624 tr

## 8. Chấm điểm

Mỗi câu trả lời có cả chỉ số v1.2 và v1.3:

- `drift_ext`: lỗi thời theo mã (chỉ số chính, như notebook 05);
- `label_drift`: dùng tên tactic cũ;
- `inconsistent`, `outcome13`, `stale13`: theo v1.3.

In [13]:
rows = []
for m in MODELS:
    for cond in CONDITIONS:
        p = RAW / m["tag"] / f"{cond}.jsonl"
        got = [json.loads(l) for l in open(p, encoding="utf-8")]
        assert [r["query_id"] for r in got] == [q["query_id"] for q in QUERIES], f"{p}: thiếu hoặc sai thứ tự"
        for r in got:
            q = QBYID[r["query_id"]]
            s = score_v13(q, r["text"], TACTIC_ALIAS)
            rows.append(dict(model=m["tag"], condition=cond, query_id=q["query_id"], base_id=q["base_id"],
                             qform=q["qform"], item_id=q["item_id"], lineage=q["lineage"],
                             revoked=ITEMS[q["item_id"]]["anchor_status_new"] != "active",
                             template=q["template"], group=q["group"], answer=r["text"],
                             finish_reason=r["finish_reason"], pred="|".join(s["pred"]), gold="|".join(q["gold"]),
                             other_gold="|".join(q["other_gold"]), em=s["em"], outcome=s["outcome"],
                             drift_ext=s["drift_ext"], label_drift=s["label_drift"], inconsistent=s["inconsistent"],
                             outcome13=s["outcome13"], stale13=s["stale13"], format=s["format"]))
SC = pd.DataFrame(rows)
SC.to_csv(OUT / "scored.csv", index=False)
chk = SC.groupby(["model", "condition"]).agg(n=("query_id", "count"), json=("format", lambda x: (x == "json").mean()),
                                             truncated=("finish_reason", lambda x: (x == "length").sum()))
assert (chk.n == len(QUERIES)).all()
chk

n  json  truncated
model       condition                           
llama3.1-8b bridge_new_old  624   1.0          0
            bridge_old_new  624   1.0          0
            new_old         624   1.0          0
            old_new         624   1.0          0
            swap_new_old    624   1.0          0
            swap_old_new    624   1.0          0
            tag_new_old     624   1.0          0
            tag_old_new     624   1.0          0
            vtag_new_old    624   1.0          0
            vtag_old_new    624   1.0          0
qwen3-8b    bridge_new_old  624   1.0          0
            bridge_old_new  624   1.0          0
            new_old         624   1.0          0
            old_new         624   1.0          0
            swap_new_old    624   1.0          0
            swap_old_new    624   1.0          0
            tag_new_old     624   1.0          0
            tag_old_new     624   1.0          0
            vtag_new_old    624   1.0          0
            vtag_old_new    624   1.0          0

## 9. Kết quả

Khoảng tin cậy 95% là bootstrap theo cụm dòng dõi (2.000 lần lặp). So sánh giữa hai điều kiện, hoặc giữa hai cách hỏi, là **ghép cặp** trên cùng câu gốc.

### 9.1. Kiểm tra lặp lại notebook 05

Bốn điều kiện `old_new`, `new_old`, `tag_old_new`, `tag_new_old` với cách hỏi `ver` có prompt giống hệt notebook 05. Bảng này so sánh:

- tỉ lệ lỗi thời ở hai lần chạy;
- tỉ lệ câu trả lời **giống hệt từng ký tự**.

vLLM chạy theo lô, và lô lần này khác notebook 05 (trộn câu `ver`/`nover` thay vì `current`/`unversioned`). Vì vậy câu trả lời có thể khác nhau ở vài câu; đó là bình thường. Điều cần xem là **tỉ lệ lỗi thời** của hai lần chạy có gần nhau không. Nếu chênh nhau rõ rệt (ví dụ hơn 0,05) thì báo thầy.

In [14]:
def sel(model, cond, group, qform="ver", data=None):
    data = SC if data is None else data
    d = data[(data.model == model) & (data.condition == cond) & (data.group == group) & (data.qform == qform)]
    return d.sort_values("base_id")

if "05" in RESCORED:
    S5 = RESCORED["05"]
    rep = []
    for m in [x["tag"] for x in MODELS]:
        for c in REPLICATED:
            a = SC[(SC.model == m) & (SC.condition == c) & (SC.qform == "ver")].set_index("base_id")
            b = S5[(S5.model == m) & (S5.condition == c) & (S5.framing == "current")].set_index("query_id")
            j = a.join(b[["answer", "drift_ext"]], rsuffix="_05", how="inner")
            ch = j[j.group == "changed"]
            rep.append({"model": m, "condition": c, "n": len(j),
                        "giống hệt câu trả lời": round((j.answer == j.answer_05).mean(), 3),
                        "lỗi thời changed (06)": round(ch.drift_ext.mean(), 3),
                        "lỗi thời changed (05)": round(ch.drift_ext_05.mean(), 3)})
    T_REP = pd.DataFrame(rep)
    T_REP.to_csv(OUT / "replication_05.csv", index=False)
    display(T_REP)
else:
    print("(Không có kết quả notebook 05 — bỏ qua)")

,model,condition,n,giống hệt câu trả lời,lỗi thời changed (06),lỗi thời changed (05)
0,qwen3-8b,old_new,312,1.00,0.938,0.938
1,qwen3-8b,new_old,312,1.00,0.839,0.839
2,qwen3-8b,tag_old_new,312,1.00,0.080,0.080
3,qwen3-8b,tag_new_old,312,1.00,0.268,0.268
4,llama3.1-8b,old_new,312,0.99,0.929,0.929
5,llama3.1-8b,new_old,312,0.99,0.768,0.777
6,llama3.1-8b,tag_old_new,312,0.99,0.054,0.054
7,llama3.1-8b,tag_new_old,312,1.00,0.223,0.223


### 9.2. Bảng tổng theo điều kiện × cách hỏi

Mỗi ô ghi `lỗi thời [KTC 95%]`:

- nhóm `changed`: `drift_ext`;
- nhóm `label`: tỉ lệ dùng tên cũ.

Cột độ chính xác giúp nhận ra trường hợp lỗi thời giảm chỉ vì mô hình bỏ trống câu trả lời.

In [15]:
T1 = []
for m in [x["tag"] for x in MODELS]:
    for qf in ("ver", "nover"):
        for c in CONDITIONS:
            T1.append({"model": m, "qform": qf, "condition": c,
                       "changed: lỗi thời": fmt_b(boot_rate(sel(m, c, "changed", qf), "drift_ext")),
                       "changed: acc": fmt_b(boot_rate(sel(m, c, "changed", qf), "em")),
                       "label: tên cũ": fmt_b(boot_rate(sel(m, c, "label", qf), "label_drift")),
                       "label: acc": fmt_b(boot_rate(sel(m, c, "label", qf), "em")),
                       "stable: acc": fmt_b(boot_rate(sel(m, c, "stable", qf), "em"))})
T1 = pd.DataFrame(T1)
T1.to_csv(OUT / "summary_conditions.csv", index=False)
T1

,model,qform,condition,changed: lỗi thời,changed: acc,label: tên cũ,label: acc,stable: acc
0,qwen3-8b,ver,old_new,"0.94 [0.89, 0.98]","0.06 [0.02, 0.11]","0.97 [0.94, 1.00]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
1,qwen3-8b,ver,new_old,"0.84 [0.76, 0.91]","0.16 [0.09, 0.24]","0.95 [0.90, 0.99]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
2,qwen3-8b,ver,tag_old_new,"0.08 [0.02, 0.15]","0.91 [0.84, 0.98]","0.05 [0.01, 0.10]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
3,qwen3-8b,ver,tag_new_old,"0.27 [0.15, 0.38]","0.71 [0.60, 0.83]","0.34 [0.24, 0.45]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
4,qwen3-8b,ver,vtag_old_new,"0.41 [0.30, 0.51]","0.59 [0.49, 0.70]","0.29 [0.19, 0.39]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
5,qwen3-8b,ver,vtag_new_old,"0.57 [0.46, 0.67]","0.43 [0.33, 0.54]","0.78 [0.69, 0.86]","0.97 [0.94, 1.00]","1.00 [1.00, 1.00]"
6,qwen3-8b,ver,swap_old_new,"0.99 [0.97, 1.00]","0.00 [0.00, 0.00]","1.00 [1.00, 1.00]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
7,qwen3-8b,ver,swap_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]","0.99 [0.96, 1.00]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"
8,qwen3-8b,ver,bridge_old_new,"0.00 [0.00, 0.00]","1.00 [1.00, 1.00]","0.05 [0.01, 0.10]","1.00 [1.00, 1.00]","1.00 [1.00, 1.00]"
9,qwen3-8b,ver,bridge_new_old,"0.09 [0.04, 0.15]","0.89 [0.83, 0.95]","0.34 [0.24, 0.45]","0.99 [0.96, 1.00]","1.00 [1.00, 1.00]"


### 9.3. Kiểm định M1–M5

**Cách đọc cột "hỗ trợ":**

- **"✓ (>0)"**: khoảng tin cậy nằm hẳn trên 0, đúng chiều dự đoán.
- **"ngược chiều"**: khoảng tin cậy nằm hẳn dưới 0.

**Phạm vi dữ liệu:**

- M1–M4b được tính cho cả nhóm `changed` (chỉ số `drift_ext`) và nhóm `label` (chỉ số tên cũ).
- Các so sánh giữa hai cách hỏi (M2, M2v, M3, M4n) lấy **D1 + D2** làm kết quả chính. Dòng "mọi mẫu" chỉ để tham khảo (xem lưu ý về D4 ở đầu notebook).
- M5 chỉ xét các câu `changed` có mã bị thu hồi. Bảng có hai dòng: giảm lỗi thời và tăng độ chính xác.

In [16]:
REV = {"old_new": "new_old", "new_old": "old_new"}
NO_D4 = SC[SC.template != "D4_id_currency"]          # dữ liệu chính cho các so sánh giữa hai cách hỏi
# (tên, thứ tự X, (điều kiện a, cách hỏi a), (điều kiện b, cách hỏi b), dữ liệu chính, có báo thêm "mọi mẫu" không)
MECH = []
for X in ("old_new", "new_old"):
    MECH += [
        ("M1 chữ current/superseded",           X, ("vtag_" + X, "ver"),   ("tag_" + X, "ver"),        "all"),
        ("M2 số phiên bản trong câu hỏi",        X, ("tag_" + X, "nover"),  ("tag_" + X, "ver"),        "noD4"),
        ("M2v khớp chữ v19.2 (nhãn chỉ có số)",  X, ("vtag_" + X, "nover"), ("vtag_" + X, "ver"),       "noD4"),
        ("M3 số phiên bản trần, không gợi ý",    X, (X, "nover"),           ("vtag_" + X, "nover"),     "noD4"),
        ("M4 theo nhãn (giữ thứ tự nhãn)",       X, ("swap_" + X, "ver"),   ("tag_" + REV[X], "ver"),   "all"),
        ("M4n như M4, câu hỏi không số",         X, ("swap_" + X, "nover"), ("tag_" + REV[X], "nover"), "noD4"),
        ("M4-alt tráo nhãn, giữ thứ tự nội dung", X, ("swap_" + X, "ver"),   ("tag_" + X, "ver"),        "all"),
        ("M4b nhãn sai so với không nhãn",       X, ("swap_" + X, "ver"),   (X, "ver"),                 "all"),
    ]

def mech_table(metric_changed="drift_ext", metric_label="label_drift"):
    out = []
    for m in [x["tag"] for x in MODELS]:
        for name, X, (ca, qa), (cb, qb), scope in MECH:
            scopes = [("D1+D2", NO_D4), ("mọi mẫu (tham khảo)", SC)] if scope == "noD4" else [("mọi mẫu", SC)]
            for sc_name, data in scopes:
                for grp, col in (("changed", metric_changed), ("label", metric_label)):
                    d = boot_diff_paired(sel(m, ca, grp, qa, data), sel(m, cb, grp, qb, data), col)
                    out.append({"model": m, "giả thuyết": name, "thứ tự": X, "dữ liệu": sc_name, "nhóm": grp,
                                "so sánh": f"{col}({ca},{qa}) − {col}({cb},{qb})", "hiệu": fmt_b(d),
                                "hỗ trợ": verdict(d)})
        for grp, col in (("changed", metric_changed), ("label", metric_label)):
            a = pd.concat([sel(m, "swap_old_new", grp), sel(m, "swap_new_old", grp)])
            b = pd.concat([sel(m, "tag_new_old", grp), sel(m, "tag_old_new", grp)])
            d = boot_diff_paired(a, b, col)
            out.append({"model": m, "giả thuyết": "M4 gộp hai thứ tự (kết luận chính)", "thứ tự": "cả hai",
                        "dữ liệu": "mọi mẫu", "nhóm": grp, "so sánh": f"{col}(swap_*) − {col}(tag_*), ver",
                        "hiệu": fmt_b(d), "hỗ trợ": verdict(d)})
        rv = SC[SC.revoked]
        for X in ("old_new", "new_old"):
            a, b = sel(m, "tag_" + X, "changed", "ver", rv), sel(m, "bridge_" + X, "changed", "ver", rv)
            d, acc = boot_diff_paired(a, b, metric_changed), boot_diff_paired(b, a, "em")
            out.append({"model": m, "giả thuyết": "M5 cầu nối: giảm lỗi thời", "thứ tự": X, "dữ liệu": "mã thu hồi",
                        "nhóm": "changed", "so sánh": f"{metric_changed}(tag_{X}) − {metric_changed}(bridge_{X})",
                        "hiệu": fmt_b(d), "hỗ trợ": verdict(d)})
            out.append({"model": m, "giả thuyết": "M5 cầu nối: tăng độ chính xác", "thứ tự": X, "dữ liệu": "mã thu hồi",
                        "nhóm": "changed", "so sánh": f"acc(bridge_{X}) − acc(tag_{X})",
                        "hiệu": fmt_b(acc), "hỗ trợ": verdict(acc)})
    return pd.DataFrame(out)

T2 = mech_table()
T2.to_csv(OUT / "summary_mechanisms.csv", index=False)
T2

,model,giả thuyết,thứ tự,dữ liệu,nhóm,so sánh,hiệu,hỗ trợ
0,qwen3-8b,M1 chữ current/superseded,old_new,mọi mẫu,changed,"drift_ext(vtag_old_new,ver) − drift_ext(tag_ol...","0.33 [0.24, 0.42]",✓ (>0)
1,qwen3-8b,M1 chữ current/superseded,old_new,mọi mẫu,label,"label_drift(vtag_old_new,ver) − label_drift(ta...","0.24 [0.15, 0.34]",✓ (>0)
2,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,D1+D2,changed,"drift_ext(tag_old_new,nover) − drift_ext(tag_o...","0.05 [0.01, 0.10]",✓ (>0)
3,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,D1+D2,label,"label_drift(tag_old_new,nover) − label_drift(t...","0.02 [-0.02, 0.08]",·
4,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,mọi mẫu (tham khảo),changed,"drift_ext(tag_old_new,nover) − drift_ext(tag_o...","0.04 [0.00, 0.08]",·
...,...,...,...,...,...,...,...,...
103,llama3.1-8b,M4 gộp hai thứ tự (kết luận chính),cả hai,mọi mẫu,label,"label_drift(swap_*) − label_drift(tag_*), ver","0.78 [0.71, 0.84]",✓ (>0)
104,llama3.1-8b,M5 cầu nối: giảm lỗi thời,old_new,mã thu hồi,changed,drift_ext(tag_old_new) − drift_ext(bridge_old_...,"0.05 [0.00, 0.11]",·
105,llama3.1-8b,M5 cầu nối: tăng độ chính xác,old_new,mã thu hồi,changed,acc(bridge_old_new) − acc(tag_old_new),"0.15 [0.03, 0.24]",✓ (>0)
106,llama3.1-8b,M5 cầu nối: giảm lỗi thời,new_old,mã thu hồi,changed,drift_ext(tag_new_old) − drift_ext(bridge_new_...,"0.25 [0.10, 0.38]",✓ (>0)


### 9.4. Phân tích độ nhạy: chấm chặt bằng v1.3

Cùng các so sánh như 9.3, nhưng dùng `stale13` cho cả hai nhóm. Nếu kết luận đổi chiều so với 9.3, ta phải nói rõ trong bài.

In [17]:
T2s = mech_table("stale13", "stale13")
T2s.to_csv(OUT / "summary_mechanisms_strict.csv", index=False)
T2s[["model", "giả thuyết", "thứ tự", "dữ liệu", "nhóm", "hiệu", "hỗ trợ"]]

,model,giả thuyết,thứ tự,dữ liệu,nhóm,hiệu,hỗ trợ
0,qwen3-8b,M1 chữ current/superseded,old_new,mọi mẫu,changed,"0.33 [0.24, 0.42]",✓ (>0)
1,qwen3-8b,M1 chữ current/superseded,old_new,mọi mẫu,label,"0.24 [0.15, 0.34]",✓ (>0)
2,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,D1+D2,changed,"0.05 [0.01, 0.10]",✓ (>0)
3,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,D1+D2,label,"0.02 [-0.02, 0.08]",·
4,qwen3-8b,M2 số phiên bản trong câu hỏi,old_new,mọi mẫu (tham khảo),changed,"0.04 [0.00, 0.08]",·
...,...,...,...,...,...,...,...
103,llama3.1-8b,M4 gộp hai thứ tự (kết luận chính),cả hai,mọi mẫu,label,"0.78 [0.71, 0.84]",✓ (>0)
104,llama3.1-8b,M5 cầu nối: giảm lỗi thời,old_new,mã thu hồi,changed,"0.05 [0.00, 0.11]",·
105,llama3.1-8b,M5 cầu nối: tăng độ chính xác,old_new,mã thu hồi,changed,"0.15 [0.03, 0.24]",✓ (>0)
106,llama3.1-8b,M5 cầu nối: giảm lỗi thời,new_old,mã thu hồi,changed,"0.25 [0.10, 0.38]",✓ (>0)


### 9.5. Theo mẫu câu hỏi × mã bị thu hồi (nhóm `changed`, cách hỏi `ver`)

Mỗi ô ghi `lỗi thời / acc (n)`. Bảng này giúp đọc M4 và M5: phần lỗi thời còn lại tập trung ở đâu. Với M5, so sánh dòng `tag_*` với dòng `bridge_*` của từng mẫu có `revoked = True`. D4 cần đọc thận trọng, vì ghi chú cầu nối gần như nói thẳng đáp án.

In [18]:
CH = SC[(SC.group == "changed") & (SC.qform == "ver")]
_rows = [dict(template=t, revoked=r, condition=c, model=m,
              cell=f"{g.drift_ext.mean():.2f} / {g.em.mean():.2f} (n={len(g)})")
         for (t, r, c, m), g in CH.groupby(["template", "revoked", "condition", "model"])]
T3 = (pd.DataFrame(_rows).pivot(index=["template", "revoked", "condition"], columns="model", values="cell")
      .reindex(list(CONDITIONS), level="condition"))
T3.to_csv(OUT / "summary_by_template.csv")
T3

model                                            llama3.1-8b  \
template          revoked condition                            
D1_tech_tactic    False   old_new         0.93 / 0.07 (n=72)   
                          new_old         0.68 / 0.32 (n=72)   
                          tag_old_new     0.04 / 0.96 (n=72)   
                          tag_new_old     0.04 / 0.96 (n=72)   
                          vtag_old_new    0.11 / 0.89 (n=72)   
                          vtag_new_old    0.46 / 0.54 (n=72)   
                          swap_old_new    0.99 / 0.01 (n=72)   
                          swap_new_old    1.00 / 0.00 (n=72)   
                          bridge_old_new  0.04 / 0.96 (n=72)   
                          bridge_new_old  0.04 / 0.96 (n=72)   
                  True    old_new         1.00 / 0.00 (n=15)   
                          new_old         1.00 / 0.00 (n=15)   
                          tag_old_new     0.13 / 0.87 (n=15)   
                          tag_new_old     1.00 / 0.00 (n=15)   
                          vtag_old_new    0.33 / 0.67 (n=15)   
                          vtag_new_old    1.00 / 0.00 (n=15)   
                          swap_old_new    1.00 / 0.00 (n=15)   
                          swap_new_old    1.00 / 0.00 (n=15)   
                          bridge_old_new  0.07 / 0.93 (n=15)   
                          bridge_new_old  0.80 / 0.20 (n=15)   
D2_subtech_parent True    old_new          1.00 / 0.00 (n=8)   
                          new_old          0.88 / 0.00 (n=8)   
                          tag_old_new      0.12 / 0.25 (n=8)   
                          tag_new_old      0.38 / 0.00 (n=8)   
                          vtag_old_new     0.38 / 0.00 (n=8)   
                          vtag_new_old     0.38 / 0.00 (n=8)   
                          swap_old_new     1.00 / 0.00 (n=8)   
                          swap_new_old     1.00 / 0.00 (n=8)   
                          bridge_old_new   0.00 / 0.88 (n=8)   
                          bridge_new_old   0.00 / 0.25 (n=8)   
D4_id_currency    True    old_new         0.82 / 0.06 (n=17)   
                          new_old         0.88 / 0.00 (n=17)   
                          tag_old_new     0.00 / 1.00 (n=17)   
                          tag_new_old     0.24 / 0.71 (n=17)   
                          vtag_old_new    0.00 / 0.94 (n=17)   
                          vtag_new_old    0.53 / 0.41 (n=17)   
                          swap_old_new    0.88 / 0.00 (n=17)   
                          swap_new_old    1.00 / 0.00 (n=17)   
                          bridge_old_new  0.00 / 1.00 (n=17)   
                          bridge_new_old  0.00 / 1.00 (n=17)   

model                                               qwen3-8b  
template          revoked condition                           
D1_tech_tactic    False   old_new         0.90 / 0.10 (n=72)  
                          new_old         0.76 / 0.24 (n=72)  
                          tag_old_new     0.00 / 1.00 (n=72)  
                          tag_new_old     0.10 / 0.88 (n=72)  
                          vtag_old_new    0.28 / 0.72 (n=72)  
                          vtag_new_old    0.38 / 0.62 (n=72)  
                          swap_old_new    0.99 / 0.00 (n=72)  
                          swap_new_old    1.00 / 0.00 (n=72)  
                          bridge_old_new  0.00 / 1.00 (n=72)  
                          bridge_new_old  0.10 / 0.88 (n=72)  
                  True    old_new         1.00 / 0.00 (n=15)  
                          new_old         1.00 / 0.00 (n=15)  
                          tag_old_new     0.27 / 0.73 (n=15)  
                          tag_new_old     0.93 / 0.07 (n=15)  
                          vtag_old_new    0.73 / 0.27 (n=15)  
                          vtag_new_old    1.00 / 0.00 (n=15)  
                          swap_old_new    1.00 / 0.00 (n=15)  
                          swap_new_old    1.00 / 0.00 (n=15)  
                          bridge_old_new  0.00 / 1.

### 9.6. Đối chứng: nhóm `stable`

Ở nhóm `stable`, đoạn O và N có cùng đáp án. Tráo nhãn hay đổi kiểu nhãn **không nên** làm giảm độ chính xác ở đây. Nếu có giảm, đó là tác dụng phụ của nhãn chứ không phải do trôi phiên bản.

In [19]:
T4 = []
for m in [x["tag"] for x in MODELS]:
    for qf in ("ver", "nover"):
        for c in CONDITIONS:
            if c == "old_new":
                continue
            T4.append({"model": m, "qform": qf, "condition": c,
                       "stable acc": fmt_b(boot_rate(sel(m, c, "stable", qf), "em")),
                       "Δ so với old_new": fmt_b(boot_diff_paired(sel(m, c, "stable", qf), sel(m, "old_new", "stable", qf), "em"))})
pd.DataFrame(T4)

,model,qform,condition,stable acc,Δ so với old_new
0,qwen3-8b,ver,new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
1,qwen3-8b,ver,tag_old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
2,qwen3-8b,ver,tag_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
3,qwen3-8b,ver,vtag_old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
4,qwen3-8b,ver,vtag_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
5,qwen3-8b,ver,swap_old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
6,qwen3-8b,ver,swap_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
7,qwen3-8b,ver,bridge_old_new,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
8,qwen3-8b,ver,bridge_new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"
9,qwen3-8b,nover,new_old,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]"


### 9.7. Phân bố kết quả v1.3 (nhóm `changed`, cách hỏi `ver`)

In [20]:
T5 = pd.crosstab([CH.model, CH.condition], CH.outcome13, normalize="index").round(3)
T5.to_csv(OUT / "summary_outcomes13.csv")
T5

outcome13                   CORRECT  INCONSISTENT  PARTIAL  STALE  WRONG
model       condition                                                   
llama3.1-8b bridge_new_old    0.804         0.009    0.045  0.134  0.009
            bridge_old_new    0.955         0.000    0.000  0.036  0.009
            new_old           0.196         0.018    0.027  0.759  0.000
            old_new           0.009         0.018    0.018  0.955  0.000
            swap_new_old      0.000         0.000    0.000  1.000  0.000
            swap_old_new      0.000         0.000    0.009  0.982  0.009
            tag_new_old       0.714         0.009    0.054  0.223  0.000
            tag_old_new       0.902         0.000    0.036  0.054  0.009
            vtag_new_old      0.411         0.009    0.054  0.527  0.000
            vtag_old_new      0.795         0.009    0.054  0.143  0.000
qwen3-8b    bridge_new_old    0.893         0.018    0.000  0.089  0.000
            bridge_old_new    1.000         0.000    0.000  0.000  0.000
            new_old           0.125         0.000    0.000  0.875  0.000
            old_new           0.054         0.054    0.000  0.893  0.000
            swap_new_old      0.000         0.000    0.000  1.000  0.000
            swap_old_new      0.000         0.000    0.009  0.991  0.000
            tag_new_old       0.714         0.018    0.000  0.268  0.000
            tag_old_new       0.911         0.000    0.009  0.080  0.000
            vtag_new_old      0.429         0.000    0.000  0.571  0.000
            vtag_old_new      0.589         0.000    0.000  0.411  0.000

## 10. Ghi manifest và đóng gói gửi thầy

In [21]:
MANIFEST["n_queries"] = len(QUERIES)
MANIFEST["scorer_v13_sha256"] = sha256(ROOT / "ctidrift_scoring_v13.py")
MANIFEST["runs"] = {}
for m in MODELS:
    for cond in CONDITIONS:
        mp = RAW / m["tag"] / f"meta_{cond}.json"
        if mp.exists():
            MANIFEST["runs"][f"{m['tag']}/{cond}"] = json.loads(mp.read_text())
MANIFEST["outputs"] = {str(p.relative_to(OUT)): sha256(p) for p in sorted(OUT.rglob("*"))
                       if p.is_file() and p.suffix in (".csv", ".jsonl") and "raw_smoke" not in p.parts
                       and not p.parts[len(OUT.parts)].startswith("prompts")}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
ZIP = OUT / "gui_thay.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != ZIP and not p.parts[len(OUT.parts)].startswith("prompts_smoke"):
            z.write(p, p.relative_to(OUT))
    z.write(ROOT / "ctidrift_scoring_v13.py", "ctidrift_scoring_v13.py")
print(f"Đã tạo {ZIP.relative_to(ROOT)} ({ZIP.stat().st_size / 1e6:.1f} MB)")

Đã tạo results/mech-18.1-19.2/gui_thay.zip (3.2 MB)


## 11. Gửi thầy

1. Notebook đã chạy: `06_co_che_nhan_phien_ban.ipynb`.
2. File `results/mech-18.1-19.2/gui_thay.zip`, gồm prompt, câu trả lời thô, `scored.csv`, các bảng kết quả và bộ chấm v1.3.

Kết quả M1–M5 sẽ cho biết phương pháp của Trụ 3 cần những thành phần nào: nhãn phiên bản dạng nào, có cần xác định "câu hỏi đang hỏi bản nào" không, và có cần ghi chú cầu nối cho mã bị thu hồi không. M4 còn cho biết phương pháp đó có **rủi ro bảo mật** nếu nhãn bị sai hoặc bị sửa.